In [6]:
#!pip install facenet-pytorch opencv-python torch torchvision

In [5]:
import torch
from facenet_pytorch import MTCNN, InceptionResnetV1
import cv2
from PIL import Image
import numpy as np
import os

# Check if GPU is available
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# mtcnn: for detection; resnet: for recognition (FaceNet)
mtcnn = MTCNN(keep_all=True, device=device)
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

In [7]:
def build_database(data_path):
    database = {}
    for person_name in os.listdir(data_path):
        person_dir = os.path.join(data_path, person_name)
        embeddings = []
        for img_name in os.listdir(person_dir):
            img = Image.open(os.path.join(person_dir, img_name))
            # MTCNN crops and resizes face automatically
            face = mtcnn(img)
            if face is not None:
                # Generate embedding (1st face in image)
                emb = resnet(face[0].unsqueeze(0).to(device)).detach().cpu().numpy()
                embeddings.append(emb)

        # Store the average embedding for the person for stability
        if embeddings:
            database[person_name] = np.mean(embeddings, axis=0)
    return database

# Usage:
# known_faces = build_database('path_to_team_images')

In [8]:
def live_recognition(database, threshold=0.8):
    cap = cv2.VideoCapture(0)

    while True:
        ret, frame = cap.read()
        img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

        # Detect all faces in frame
        boxes, _ = mtcnn.detect(img)

        if boxes is not None:
            faces = mtcnn(img)
            for i, box in enumerate(boxes):
                # Get embedding for detected face
                face_emb = resnet(faces[i].unsqueeze(0).to(device)).detach().cpu().numpy()

                # Compare with database
                min_dist = 100
                identity = "Unknown"

                for name, db_emb in database.items():
                    dist = np.linalg.norm(face_emb - db_emb) # Euclidean Distance
                    if dist < min_dist:
                        min_dist = dist
                        identity = name if dist < threshold else "Unknown"

                # Draw GUI elements
                x, y, w, h = box.astype(int)
                color = (0, 255, 0) if identity != "Unknown" else (0, 0, 255)
                cv2.rectangle(frame, (x, y), (w, h), color, 2)
                cv2.putText(frame, f"{identity} ({min_dist:.2f})", (x, y-10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        cv2.imshow('Lab S2-B5C-02: Face Recognition', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'): break

    cap.release()
    cv2.destroyAllWindows()